# 02 – Seleção e limpeza

**Objetivo:** aplicar os tratamentos definidos no diagnóstico do notebook 01, por meio das funções
documentadas em `src/funcoes.py`. Cada etapa mostra o estado dos dados antes e depois do tratamento.
Ao final, o dataset limpo é salvo para as análises do notebook 03.

In [16]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from src.funcoes import (
    carregar_dados, selecionar_colunas, renomear_colunas,
    converter_para_numerico, preencher_faltantes, marcar_como_faltante
)

pd.set_option("display.max_columns", None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
df_bruto = carregar_dados("../data/spotify-2023.csv")

Dados carregados: 953 linhas e 24 colunas.


## Etapa 1 – Seleção de colunas
Mantive apenas as 19 colunas relacionadas ao Spotify. As colunas de Apple Music,
Deezer e Shazam ficam fora do escopo; com isso, também saem `in_deezer_playlists` e
`in_shazam_charts`, que tinham tipos incorretos e valores faltantes.

In [18]:
colunas_spotify = [
    "track_name", "artist(s)_name", "artist_count",
    "released_year", "released_month", "released_day",
    "streams", "in_spotify_playlists", "in_spotify_charts",
    "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%", "acousticness_%",
    "instrumentalness_%", "liveness_%", "speechiness_%",
]

df = selecionar_colunas(df_bruto, colunas_spotify)
print(f"Antes: {df_bruto.shape} | Depois: {df.shape}")

Antes: (953, 24) | Depois: (953, 19)


In [19]:
try:
    selecionar_colunas(df_bruto, ["track_name", "coluna_que_nao_existe"])
    print("FALHOU: a função deveria ter levantado um erro.")
except KeyError as erro:
    print(f"Teste OK – erro capturado: {erro}")

Teste OK – erro capturado: "Colunas não encontradas no DataFrame: ['coluna_que_nao_existe']"


## Etapa 2 – Renomeação de colunas
Removi símbolos dos nomes (`(s)`, `%`), que dificultam a escrita do código.
As demais colunas já estão em um formato adequado e mantêm o nome original.

In [20]:
novos_nomes = {
    "artist(s)_name": "artist_name",
    "danceability_%": "danceability",
    "valence_%": "valence",
    "energy_%": "energy",
    "acousticness_%": "acousticness",
    "instrumentalness_%": "instrumentalness",
    "liveness_%": "liveness",
    "speechiness_%": "speechiness",
}

df = renomear_colunas(df, novos_nomes)
df.columns.tolist()

['track_name',
 'artist_name',
 'artist_count',
 'released_year',
 'released_month',
 'released_day',
 'streams',
 'in_spotify_playlists',
 'in_spotify_charts',
 'bpm',
 'key',
 'mode',
 'danceability',
 'valence',
 'energy',
 'acousticness',
 'instrumentalness',
 'liveness',
 'speechiness']

## Etapa 3 – Conversão de `streams` para número
`streams` foi lida como texto por causa de um único valor corrompido (linha 574, "Love Grows").
Esse valor vira `NaN`; a linha é mantida, pois as demais colunas estão normais.

In [21]:
print(f"Tipo antes: {df['streams'].dtype}")
print(f"Linha 574 antes: {df.loc[574, 'streams']}\n")

df = converter_para_numerico(df, "streams")

print(f"\nTipo depois: {df['streams'].dtype}")
print(f"Linha 574 depois: {df.loc[574, 'streams']}")

Tipo antes: str
Linha 574 antes: BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3

Aviso: 1 valor(es) de 'streams' não eram numéricos e viraram NaN.

Tipo depois: float64
Linha 574 depois: nan


In [22]:
# Checagem

df["streams"].describe()

count    9.520000e+02
mean     5.141374e+08
std      5.668569e+08
min      2.762000e+03
25%      1.416362e+08
50%      2.905309e+08
75%      6.738690e+08
max      3.703895e+09
Name: streams, dtype: float64

**Conferindo:** O valor mínimo de 2762 streams é estranho, considerando que o primeiro quartil é de 141636175.0. Dado isso, vou conferir os 5 menores valores:

In [23]:
colunas_ver = ["track_name", "artist_name", "released_year", "released_month",
               "released_day", "streams", "in_spotify_playlists", "in_spotify_charts"]
df.nsmallest(5, "streams")[colunas_ver]

,track_name,artist_name,released_year,released_month,released_day,streams,in_spotify_playlists,in_spotify_charts
123,Que Vuelvas,"Carin Leon, Grupo Frontera",2022,12,9,2762.0,763,26
393,Jhoome Jo Pathaan,"Arijit Singh, Vishal Dadlani, Sukriti Kakar, V...",2022,12,22,1365184.0,138,4
144,QUEMA,"Sog, Ryan Castro, Peso Pluma",2023,7,13,11599388.0,437,31
142,"Gol Bolinha, Gol Quadrado 2","Mc Pedrinho, DJ 900",2023,6,1,11956641.0,293,8
68,Overdrive,Post Malone,2023,7,14,14780425.0,410,36


### Valor inválido em `streams`: "Que Vuelvas"
Após a conversão, o mínimo de `streams` é 2.762, cerca de 500 vezes menor que o segundo menor
valor (1,37 mi). O valor é incoerente com as demais medidas de sucesso da música (763 playlists,
presença nos charts) e com a data de lançamento (09/12/2022, meses antes da coleta). Conclusão:
o valor pode ser um erro de coleta. O valor vira `NaN` e a linha é mantida.

In [24]:
erro_streams = (
    (df["track_name"] == "Que Vuelvas")
    & (df["artist_name"] == "Carin Leon, Grupo Frontera")
)

df = marcar_como_faltante(df, "streams", erro_streams,
                          motivo="Erro de coleta: 2.762 streams, incoerente com playlists e charts.")

df.nsmallest(3, "streams")[colunas_ver]

1 valor(es) de 'streams' marcado(s) como faltante(s). Erro de coleta: 2.762 streams, incoerente com playlists e charts.


,track_name,artist_name,released_year,released_month,released_day,streams,in_spotify_playlists,in_spotify_charts
393,Jhoome Jo Pathaan,"Arijit Singh, Vishal Dadlani, Sukriti Kakar, V...",2022,12,22,1365184.0,138,4
144,QUEMA,"Sog, Ryan Castro, Peso Pluma",2023,7,13,11599388.0,437,31
142,"Gol Bolinha, Gol Quadrado 2","Mc Pedrinho, DJ 900",2023,6,1,11956641.0,293,8


In [25]:
# Checagem

df["streams"].count()

np.int64(951)

## Etapa 4 – Preenchimento dos faltantes de `key`
Os 95 fvalores faltantes da coluna `key` correspondem à nota C, o que foi conferido a partir de uma amostra
aleatória de 10 músicas: (10 de 10 em C). Por isso, optei por preencher com "C", registrando essa suposição.

In [26]:
antes = df["key"].value_counts(dropna=False)

df = preencher_faltantes(df, "key", "C")

depois = df["key"].value_counts(dropna=False)
pd.DataFrame({"antes": antes, "depois": depois}).fillna(0).astype(int)

95 valor(es) faltante(s) em 'key' preenchido(s) com 'C'.


,antes,depois
key,,
A,75,75
A#,57,57
B,81,81
C,0,95
C#,120,120
D,81,81
D#,33,33
E,62,62
F,89,89
